In [ ]:
# ==============================================================================
# 1. IMPORTS
# ==============================================================================

# OpenPIV
from openpiv import windef
from openpiv import tools, scaling, validation, filters, preprocess
import openpiv.pyprocess as process
from openpiv import pyprocess

# General
import numpy as np
import tifffile as tif
import napari
import matplotlib.pyplot as plt
import pandas as pd
from roifile import ImagejRoi

import pathlib
from time import time
import warnings
%matplotlib qt

# Custom functions
from src.PIV import run_PIV_on_frames

In [ ]:
# ==============================================================================
# Paths
# ==============================================================================

unstressed_reference_path = '/mnt/crunch/Clark/Fly_TFM/data/second/unstressed_reference/'
sitting_reference_path = '/mnt/crunch/Clark/Fly_TFM/data/second/sitting_reference/'
time_resolved_path = '/mnt/crunch/Clark/Fly_TFM/data/second/time_resolved/'

In [ ]:
# ==============================================================================
# Do subtraction using sitting reference
# ==============================================================================

starting_frame = 1
num_frames = 28
num_gridpoints = 3969 # len(sitting_reference)

u_sitting = np.zeros((num_frames, num_gridpoints))
v_sitting = np.zeros((num_frames, num_gridpoints))
    
for index, current_frame in enumerate(range(starting_frame, num_frames + 1)):
        
    # Load data
    data = pd.read_csv(sitting_reference_path + f'PIVlab_{current_frame:04d}.txt', skiprows=2)
        
    # Load columns and inject columns into subtaction method
    u_sitting[index] = data['u [px/frame]'].values #- sitting_reference['u [px/frame]'].values
    v_sitting[index] = data['v [px/frame]'].values #- sitting_reference['v [px/frame]'].values

In [ ]:
# ==============================================================================
# Do subtraction ACTUALLY using sitting reference WRT THE UNSTRESSED REFERENCE
# ==============================================================================

sitting_reference = pd.read_csv('/mnt/crunch/Clark/Fly_TFM/data/second/unstressed_reference/PIVlab_0040.txt', skiprows=2)

starting_frame = 40
num_frames = 28
num_gridpoints = len(sitting_reference)

u_subtraction = np.zeros((num_frames, num_gridpoints))
v_subtraction = np.zeros((num_frames, num_gridpoints))
    
for index, current_frame in enumerate(range(starting_frame, starting_frame + num_frames + 0)):
        
    # Load data
    data = pd.read_csv(unstressed_reference_path + f'PIVlab_{current_frame:04d}.txt', skiprows=2)
        
    # Load columns and inject columns into subtaction method
    u_subtraction[index] = data['u [px/frame]'].values - sitting_reference['u [px/frame]'].values
    v_subtraction[index] = data['v [px/frame]'].values - sitting_reference['v [px/frame]'].values
    
    print(f'Frame {current_frame} loaded and subtracted from sitting reference.')
    print(f'Progress: {index + 1}/{num_frames} frames processed.')
    print(unstressed_reference_path + f'PIVlab_{current_frame:04d}.txt')

In [ ]:
# Subtract background fit for cell below

from scipy.io import loadmat
from matplotlib.path import Path
from mpl_toolkits.mplot3d import Axes3D
import plotly.graph_objects as go

# ------------------------------------------------------------------------------
# Load mask + frame

mask_coords = loadmat('/mnt/crunch/Clark/Fly_TFM/data/second/PIVlab_mask_2.mat')['masks_in_frame'][0, 39][0, 1]
polygon = Path(mask_coords)

wave_start_tif = tif.imread('/mnt/crunch/Clark/Fly_TFM/data/second/second_best.tif')[40]

# ------------------------------------------------------------------------------
# Load PIV displacements for that frame

wave_start_displacements = pd.read_csv('/mnt/crunch/Clark/Fly_TFM/data/second/unstressed_reference/PIVlab_0040.txt', skiprows=2)

x_piv = wave_start_displacements['x [px]'].values
y_piv = wave_start_displacements['y [px]'].values
u_piv = wave_start_displacements['u [px/frame]'].values
v_piv = wave_start_displacements['v [px/frame]'].values

points = np.column_stack([x_piv, y_piv])
inside_mask = polygon.contains_points(points)
outside_mask = ~inside_mask

x_filtered = x_piv[outside_mask]
y_filtered = y_piv[outside_mask]
u_filtered = u_piv[outside_mask]
v_filtered = v_piv[outside_mask]

# ------------------------------------------------------------------------------
# Plot masked vector field

# plt.figure(figsize=(8, 8))
# plt.plot(mask_coords[:, 0], mask_coords[:, 1], 'r-', linewidth=2, label='Mask')
# plt.quiver(x_filtered, y_filtered, u_filtered, -v_filtered, color='blue', scale=100)
# plt.legend()
# plt.gca().invert_yaxis()
# plt.title('far-field vectors')
# plt.show()

# ------------------------------------------------------------------------------
# Fit quadratic surface to far-field u, v

from scipy.optimize import least_squares

def design_matrix(x, y):
    return np.column_stack([
        np.ones_like(x), x, y, x**2, y**2, x*y,
        x**3, y**3, x**2*y, x*y**2
    ])

def resid_fn(coeffs, x, y, val):
    return design_matrix(x, y) @ coeffs - val

def fit_quadratic(x, y, val, mask, loss='soft_l1'):
    x0 = np.zeros(design_matrix(x[:1], y[:1]).shape[1])
    result = least_squares(resid_fn, x0, loss=loss, args=(x[mask], y[mask], val[mask]))
    return result.x

def eval_quadratic(coeffs, x, y):
    shape = x.shape
    return (design_matrix(x.ravel(), y.ravel()) @ coeffs).reshape(shape)

coeffs_u = fit_quadratic(x_piv, y_piv, u_piv, outside_mask)
coeffs_v = fit_quadratic(x_piv, y_piv, v_piv, outside_mask)

# evaluate fit everywhere, including inside the excluded blob
u_fit = eval_quadratic(coeffs_u, x_piv, y_piv)
v_fit = eval_quadratic(coeffs_v, x_piv, y_piv)

# ------------------------------------------------------------------------------
# Residuals on the far-field data itself (fit-quality check)

u_fit_far = eval_quadratic(coeffs_u, x_filtered, y_filtered)
v_fit_far = eval_quadratic(coeffs_v, x_filtered, y_filtered)

resid_u = u_filtered - u_fit_far
resid_v = v_filtered - v_fit_far
resid_mag = np.hypot(resid_u, resid_v)

ss_res = np.sum(resid_u**2 + resid_v**2)
ss_tot = np.sum((u_filtered - u_filtered.mean())**2 + (v_filtered - v_filtered.mean())**2)
r2 = 1 - ss_res / ss_tot
print('R^2:', r2)

# ------------------------------------------------------------------------------
# 3D surface visualization, u and v, matplotlib

xx, yy = np.meshgrid(
    np.linspace(x_piv.min(), x_piv.max(), 50),
    np.linspace(y_piv.min(), y_piv.max(), 50)
)
zz_u = eval_quadratic(coeffs_u, xx, yy)
zz_v = eval_quadratic(coeffs_v, xx, yy)

# fig = plt.figure(figsize=(10, 8))
# ax = fig.add_subplot(111, projection='3d')
# ax.scatter(x_filtered, y_filtered, u_filtered, c='blue', s=5, label='actual (far-field)')
# ax.plot_surface(xx, yy, zz_u, alpha=0.3, color='orange')
# ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('u')
# plt.show()

# fig = plt.figure(figsize=(10, 8))
# ax = fig.add_subplot(111, projection='3d')
# ax.scatter(x_filtered, y_filtered, v_filtered, c='blue', s=5, label='actual (far-field)')
# ax.plot_surface(xx, yy, zz_v, alpha=0.3, color='orange')
# ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('v')
# plt.show()

# ------------------------------------------------------------------------------
# # Interactive plotly versions

# for label, val_filtered, zz in [('u', u_filtered, zz_u), ('v', v_filtered, zz_v)]:
#     fig = go.Figure()
#     fig.add_trace(go.Scatter3d(
#         x=x_filtered, y=y_filtered, z=val_filtered,
#         mode='markers', marker=dict(size=2, color='blue'), name='actual (far-field)'
#     ))
#     fig.add_trace(go.Surface(
#         x=xx, y=yy, z=zz, opacity=0.4, colorscale='Oranges', showscale=False, name='fit'
#     ))
#     fig.update_layout(scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title=label))
#     fig.write_html(f'{label}_surface_check.html')
    
nx = len(np.unique(x_piv))
ny = len(np.unique(y_piv))

X = x_piv.reshape(ny, nx)
Y = y_piv.reshape(ny, nx)

resid_mag_full = np.full(x_piv.shape, np.nan)
resid_mag_full[outside_mask] = resid_mag
resid_mag_grid = resid_mag_full.reshape(ny, nx)

# plt.figure(figsize=(8, 8))
# plt.pcolormesh(X, Y, resid_mag_grid, cmap='viridis', shading='auto')
# plt.colorbar(label='residual magnitude [px]')
# plt.plot(mask_coords[:, 0], mask_coords[:, 1], 'r-')
# plt.gca().invert_yaxis()
# plt.title('far-field residual (grid)')
# plt.show()

# plt.figure(figsize=(8, 8))
# plt.quiver(x_filtered, y_filtered, resid_u, -resid_v, resid_mag, cmap='viridis', scale=20)
# plt.plot(mask_coords[:, 0], mask_coords[:, 1], 'r-')
# plt.gca().invert_yaxis()
# plt.title('far-field residual vectors')
# plt.show()

In [ ]:
# ==============================================================================
# Do frame to frame subtraction using the unstressed reference
# ==============================================================================

wave_start = pd.read_csv('/mnt/crunch/Clark/Fly_TFM/data/second/unstressed_reference/PIVlab_0040.txt', skiprows=2)

starting_frame = 41
num_frames = 28
num_gridpoints = len(wave_start)

u_subtraction_frame_to_frame_unstressed = np.zeros((num_frames - 1, num_gridpoints))
v_subtraction_frame_to_frame_unstressed = np.zeros((num_frames - 1, num_gridpoints))

for index, current_frame in enumerate(range(starting_frame, starting_frame + num_frames - 1)):
    # Load data
    previous_data = pd.read_csv(unstressed_reference_path + f'PIVlab_{(current_frame - 1):04d}.txt', skiprows=2)
    current_data = pd.read_csv(unstressed_reference_path + f'PIVlab_{current_frame:04d}.txt', skiprows=2)
    
    u_subtraction_frame_to_frame_unstressed[index] = (current_data['u [px/frame]'].values - previous_data['u [px/frame]'].values)
    v_subtraction_frame_to_frame_unstressed[index] = (current_data['v [px/frame]'].values - previous_data['v [px/frame]'].values)
    
    print(f'Frame {current_frame} loaded and subtracted from previous frame.')
    print(f'Progress: {index + 1}/{num_frames} frames processed.')
    print(unstressed_reference_path + f'PIVlab_{(current_frame - 1):04d}.txt')
    print(unstressed_reference_path + f'PIVlab_{current_frame:04d}.txt')
    
print(u_subtraction_frame_to_frame_unstressed.shape)

In [ ]:
# integrate previous cell

# u_time_resolved_subtraction_frame_to_frame_unstressed = np.zeros((num_frames - 1, num_gridpoints))
# v_time_resolved_subtraction_frame_to_frame_unstressed = np.zeros((num_frames - 1, num_gridpoints))

# for index, current_frame in enumerate(range(1, num_frames)):
    
#     u_time_resolved_subtraction_frame_to_frame_unstressed[index] = 

u_integrated_subtraction_frame_to_frame_unstressed = np.cumsum(u_subtraction_frame_to_frame_unstressed, axis=0)
v_integrated_subtraction_frame_to_frame_unstressed = np.cumsum(v_subtraction_frame_to_frame_unstressed, axis=0)

In [ ]:
# now just subtract fit from every unstressed frame and plot

starting_frame = 40
num_frames = 28
num_gridpoints = len(sitting_reference)

u_subtraction_fit = np.zeros((num_frames, num_gridpoints))
v_subtraction_fit = np.zeros((num_frames, num_gridpoints))
    
for index, current_frame in enumerate(range(starting_frame, starting_frame + num_frames + 0)):
        
    # Load data
    data = pd.read_csv(unstressed_reference_path + f'PIVlab_{current_frame:04d}.txt', skiprows=2)
        
    # Load columns and inject columns into subtaction method
    u_subtraction_fit[index] = data['u [px/frame]'].values - u_fit
    v_subtraction_fit[index] = data['v [px/frame]'].values - v_fit
    
    print(f'Frame {current_frame} loaded and subtracted from sitting reference.')
    print(f'Progress: {index + 1}/{num_frames} frames processed.')
    print(unstressed_reference_path + f'PIVlab_{current_frame:04d}.txt')

In [ ]:
# # visualize above cell

# u_grid = u_subtraction_fit.reshape(28, 63, 63)
# v_grid = v_subtraction_fit.reshape(28, 63, 63)
# z_grid = np.zeros_like(u_grid) # napari expects num spatial dimensions = num vector components, so manually set dz = 0

# vector_grid = np.stack([z_grid, v_grid, u_grid], axis=-1)
# magnitude_grid = np.sqrt(u_grid**2 + v_grid**2)

# # ------------------------------------------------------------------------------

# viewer = napari.Viewer()

# magnitude_layer = viewer.add_image(
#     magnitude_grid,
#     name='Magnitude Heatmap',
#     colormap='jet',
#     #interpolation2d='bicubic',
#     contrast_limits=[0, 3]
# )

# vector_layer = viewer.add_vectors(
#     vector_grid,
#     name="Displacements",
#     vector_style='arrow',
#     edge_width=0.2,
#     length=1.0,
#     edge_color="black"
# )

# magnitude_layer.colorbar.visible = True

# napari.run()

In [ ]:
# ==============================================================================
# Integrate starting from sitting reference
# ==============================================================================

#NOTE Remember, need to add a frame manually before every other frame for visualization of integrated displacements

u_time_resolved = np.zeros((num_frames - 1, num_gridpoints))
v_time_resolved = np.zeros((num_frames - 1, num_gridpoints))

# Load .csv columns into numpy arrays
for index, current_frame in enumerate(range(1, num_frames)):
    
    # Load data
    data = pd.read_csv(time_resolved_path + f'PIVlab_{current_frame:04d}.txt', skiprows=2)
    
    # Load columns of currently integrated displacement
    u_time_resolved[index] = data['u [px/frame]'].values
    v_time_resolved[index] = data['v [px/frame]'].values
    
# Integrate
u_integrated = np.cumsum(u_time_resolved, axis=0)
v_integrated = np.cumsum(v_time_resolved, axis=0)

print(u_integrated.shape)

In [ ]:
# Load raw data

u_raw = np.zeros((num_frames, num_gridpoints))
v_raw = np.zeros((num_frames, num_gridpoints))

for index, current_frame in enumerate(range(40, 68)):
    # Load data
    data = pd.read_csv(unstressed_reference_path + f'PIVlab_{current_frame:04d}.txt', skiprows=2)
    
    # Load columns of currently integrated displacement
    u_raw[index] = data['u [px/frame]'].values
    v_raw[index] = data['v [px/frame]'].values
    
    print(f'Frame {current_frame} loaded.')
    print(f'Progress: {index + 1}/{num_frames} frames processed.')

In [ ]:
%%script true
# ==============================================================================
# Visualize the subtraction method
# ==============================================================================

u_grid = u_subtraction.reshape(28, 63, 63)
v_grid = v_subtraction.reshape(28, 63, 63)
z_grid = np.zeros_like(u_grid) # napari expects num spatial dimensions = num vector components, so manually set dz = 0

vector_grid = np.stack([z_grid, v_grid, u_grid], axis=-1)
magnitude_grid = np.sqrt(u_grid**2 + v_grid**2)

# ------------------------------------------------------------------------------

viewer = napari.Viewer()

magnitude_layer = viewer.add_image(
    magnitude_grid,
    name='Magnitude Heatmap',
    colormap='jet',
    #interpolation2d='bicubic',
    contrast_limits=[0, 3]
)

vector_layer = viewer.add_vectors(
    vector_grid,
    name="Displacements",
    vector_style='arrow',
    edge_width=0.2,
    length=1.0,
    edge_color="black"
)

magnitude_layer.colorbar.visible = True

napari.run()

In [ ]:
%%script true
# ==============================================================================
# Visualize the integration method
# ==============================================================================

u_grid = u_integrated.reshape(27, 63, 63)
v_grid = v_integrated.reshape(27, 63, 63)
z_grid = np.zeros_like(u_grid)

vector_grid = np.stack([z_grid, v_grid, u_grid], axis=-1)
magnitude_grid = np.sqrt(u_grid**2 + v_grid**2)

# ------------------------------------------------------------------------------

viewer = napari.Viewer()

magnitude_layer = viewer.add_image(
    magnitude_grid,
    name='Magnitude Heatmap',
    colormap='jet',
    #interpolation2d='bicubic',
    contrast_limits=[0, 3]
)

vector_layer = viewer.add_vectors(
    vector_grid,
    name="Displacements",
    vector_style='arrow',
    edge_width=0.2,
    length=1.0,
    edge_color="black"
)

magnitude_layer.colorbar.visible = True

napari.run()

In [ ]:
%%script true
# ==============================================================================
# Visualize the residual
# ==============================================================================

# Add a new frame to the integrated displacement arrays to match the number of frames in the subtraction method
new_row = np.zeros((1, num_gridpoints))
u_integrated_from_zero = np.vstack((new_row, u_integrated))
v_integrated_from_zero = np.vstack((new_row, v_integrated))

u_residual = u_integrated_from_zero - u_subtraction
v_residual = v_integrated_from_zero - v_subtraction

u_grid = u_residual.reshape(28, 63, 63)
v_grid = v_residual.reshape(28, 63, 63)
z_grid = np.zeros_like(u_grid)

vector_grid = np.stack([z_grid, v_grid, u_grid], axis=-1)
magnitude_grid = np.sqrt(u_grid**2 + v_grid**2)

# ------------------------------------------------------------------------------

viewer = napari.Viewer()

magnitude_layer = viewer.add_image(
    magnitude_grid,
    name='Residual Heatmap',
    colormap='jet',
    #interpolation2d='bicubic',
    contrast_limits=[0, 3]
)

vector_layer = viewer.add_vectors(
    vector_grid,
    name="Residual Displacements",
    vector_style='arrow',
    edge_width=0.2,
    length=1.0,
    edge_color="black"
)

magnitude_layer.colorbar.visible = True

napari.run()

In [ ]:
%%script true
# ==============================================================================
# 1. Prepare Integration Data (27 frames -> shifted to 28 frames)
# ==============================================================================
u_grid_integrated_raw = u_integrated.reshape(27, 63, 63)
v_grid_integrated_raw = v_integrated.reshape(27, 63, 63)
z_grid_integrated_raw = np.zeros_like(u_grid_integrated_raw)

# Prepend 1 frame of zeros to the front (axis=0) so frame 0 is empty
u_grid_integrated = np.pad(u_grid_integrated_raw, ((1, 0), (0, 0), (0, 0)), mode='constant', constant_values=0)
v_grid_integrated = np.pad(v_grid_integrated_raw, ((1, 0), (0, 0), (0, 0)), mode='constant', constant_values=0)
z_grid_integrated = np.pad(z_grid_integrated_raw, ((1, 0), (0, 0), (0, 0)), mode='constant', constant_values=0)

vector_grid_integrated = np.stack([z_grid_integrated, u_grid_integrated, v_grid_integrated], axis=-1)
magnitude_grid_integrated = np.sqrt(u_grid_integrated**2 + v_grid_integrated**2)

# ==============================================================================
# 2. Prepare Subtraction Data (28 frames)
# ==============================================================================
u_grid_subtraction = u_subtraction.reshape(28, 63, 63)
v_grid_subtraction = v_subtraction.reshape(28, 63, 63)
z_grid_subtraction = np.zeros_like(u_grid_subtraction)

vector_grid_subtraction = np.stack([z_grid_subtraction, u_grid_subtraction, v_grid_subtraction], axis=-1)
magnitude_grid_subtraction = np.sqrt(u_grid_subtraction**2 + v_grid_subtraction**2)

# RESIDUAL
# Add a new frame to the integrated displacement arrays to match the number of frames in the subtraction method
new_row = np.zeros((1, num_gridpoints))
u_integrated_from_zero = np.vstack((new_row, u_integrated))
v_integrated_from_zero = np.vstack((new_row, v_integrated))

u_residual_integrated_vs_subtraction = u_integrated_from_zero - u_subtraction
v_residual_integrated_vs_subtraction = v_integrated_from_zero - v_subtraction

u_grid_residual = u_residual_integrated_vs_subtraction.reshape(28, 63, 63)
v_grid_residual = v_residual_integrated_vs_subtraction.reshape(28, 63, 63)
z_grid_residual = np.zeros_like(u_grid_residual)

vector_grid_residual = np.stack([z_grid_residual, u_grid_residual, v_grid_residual], axis=-1)
magnitude_grid_residual = np.sqrt(u_grid_residual**2 + v_grid_residual**2)

# ==============================================================================
# 3. Launch single viewer with both datasets
# ==============================================================================
magnitude_grid_sitting = np.sqrt(u_sitting.reshape(28, 63, 63)**2 + v_sitting.reshape(28, 63, 63)**2)
vector_grid_sitting = np.stack([np.zeros_like(u_sitting.reshape(28, 63, 63)), u_sitting.reshape(28, 63, 63), v_sitting.reshape(28, 63, 63)], axis=-1)

viewer = napari.Viewer()

contrast_limits = [0, 3]

# --- Sitting Reference Layers ---
mag_sitting_layer = viewer.add_image(
    magnitude_grid_sitting,
    name='Sitting Reference - Heatmap',
    colormap='jet',
    contrast_limits=contrast_limits
)

vec_sitting_layer = viewer.add_vectors(
    vector_grid_sitting,
    name='Sitting Reference - Vectors',
    vector_style='arrow',
    edge_width=0.2,
    length=1.0,
    edge_color='black'
)

# --- Integration Method Layers ---
mag_int_layer = viewer.add_image(
    magnitude_grid_integrated,
    name='Integration - Heatmap',
    colormap='jet',
    contrast_limits=contrast_limits
)

vec_int_layer = viewer.add_vectors(
    vector_grid_integrated,
    name='Integration - Vectors',
    vector_style='arrow',
    edge_width=0.2,
    length=1.0,
    edge_color='black'
)

# --- Subtraction Method Layers ---
mag_sub_layer = viewer.add_image(
    magnitude_grid_subtraction,
    name='Subtraction - Heatmap',
    colormap='jet',
    contrast_limits=contrast_limits
)

vec_sub_layer = viewer.add_vectors(
    vector_grid_subtraction,
    name='Subtraction - Vectors',
    vector_style='arrow',
    edge_width=0.2,
    length=1.0,
    edge_color='black'
)

# --- Residual (Integrated vs Subtraction) Method Layers ---
mag_res_layer = viewer.add_image(
    magnitude_grid_residual,
    name='Residual (Integrated vs Subtraction) - Heatmap',
    colormap='jet',
    contrast_limits=contrast_limits
)

vec_res_layer = viewer.add_vectors(
    vector_grid_residual,
    name='Residual (Integrated vs Subtraction) - Vectors',
    vector_style='arrow',
    edge_width=0.2,
    length=1.0,
    edge_color='black'
)

# --- Residual (Integrated vs Sitting) Method Layers ---
u_residual_integrated_vs_sitting = u_integrated_from_zero - u_sitting
v_residual_integrated_vs_sitting = v_integrated_from_zero - v_sitting

magnitude_grid_residual_sitting = np.sqrt(u_residual_integrated_vs_sitting.reshape(28, 63, 63)**2 + v_residual_integrated_vs_sitting.reshape(28, 63, 63)**2)
vector_grid_residual_sitting = np.stack([np.zeros_like(u_residual_integrated_vs_sitting.reshape(28, 63, 63)), u_residual_integrated_vs_sitting.reshape(28, 63, 63), v_residual_integrated_vs_sitting.reshape(28, 63, 63)], axis=-1)

mag_res_sitting_layer = viewer.add_image(
    magnitude_grid_residual_sitting,
    name='Residual (Integrated vs Sitting) - Heatmap',
    colormap='jet',
    contrast_limits=contrast_limits
)

vec_res_sitting_layer = viewer.add_vectors(
    vector_grid_residual_sitting,
    name='Residual (Integrated vs Sitting) - Vectors',
    vector_style='arrow',
    edge_width=0.2,
    length=1.0,
    edge_color='black'
)

# ------------------------------------------------------------------------------

# Colorbar display
mag_sub_layer.colorbar.visible = True

# Side-by-side grid view (2 layers per box)
viewer.grid.enabled = True
viewer.grid.stride = 2

napari.run()

In [ ]:
# pad integration arrays so everything lines up (27 frames --> 28)
new_row = np.zeros((1, num_gridpoints))
u_integration = np.vstack((new_row, u_integrated))
v_integration = np.vstack((new_row, v_integrated))

u_integrated_subtraction_frame_to_frame_unstressed = np.vstack((new_row, u_integrated_subtraction_frame_to_frame_unstressed))
v_integrated_subtraction_frame_to_frame_unstressed = np.vstack((new_row, v_integrated_subtraction_frame_to_frame_unstressed))

# Residuals
u_res_int_vs_sub = u_integration - u_subtraction
v_res_int_vs_sub = v_integration - v_subtraction

u_res_int_vs_sitting = u_integration - u_sitting
v_res_int_vs_sitting = v_integration - v_sitting

u_residual_integration_vs_integration_subtraction_frame_to_frame = u_integrated_subtraction_frame_to_frame_unstressed - u_integration
v_residual_integration_vs_integration_subtraction_frame_to_frame = v_integrated_subtraction_frame_to_frame_unstressed - v_integration

datasets = [
    ("Raw Displacements",                           u_raw,        v_raw),
    #("Sitting Reference",                  u_sitting,     v_sitting),
    ("Integration Method",                 u_integration, v_integration),
    ("Integration Method (Subtraction frame to frame)", u_integrated_subtraction_frame_to_frame_unstressed, v_integrated_subtraction_frame_to_frame_unstressed),
    #("Subtraction Method",                 u_subtraction, v_subtraction),
    ("Subtraction Fit Method",             u_subtraction_fit, v_subtraction_fit),
    #("Residual (Integration - Subtraction)", u_res_int_vs_sub,     v_res_int_vs_sub),
    #("Residual (Integration - Sitting)",     u_res_int_vs_sitting, v_res_int_vs_sitting),
    ("Residual (Integration - Integration Subtraction Frame to Frame)", u_residual_integration_vs_integration_subtraction_frame_to_frame, v_residual_integration_vs_integration_subtraction_frame_to_frame)
]

# ------------------------------------------------------------------------------

viewer = napari.Viewer()

colormap='jet'
contrast_limits=[0, 3]

arrow_style = 'arrow'
arrow_edge_color = 'black'
arrow_width = 0.2
arrow_length = 1.0

scale_factor = 2048/63

for label, u_piv, v in datasets:
    u_grid = u_piv.reshape(28, 63, 63)
    v_grid = v.reshape(28, 63, 63)
    z_grid = np.zeros_like(u_grid)
    
    #u_grid = np.rot90(-u_grid, k=1, axes=(1, 2))  # Rotate the u and v grids to match the correct orientation
    #v_grid = np.rot90(v_grid, k=1, axes=(1, 2))  # Rotate the u and v grids to match the correct orientation
    
    #u_grid = np.flip(u_grid, axis=1)  # Flip the u and v grids to match the correct orientation
    #v_grid = -np.flip(v_grid, axis=1)  # Flip the u and
    
    vector_grid = np.stack([z_grid, u_grid, v_grid], axis=-1)
    magnitude_grid = np.sqrt(u_grid**2 + v_grid**2)

    viewer.add_image(magnitude_grid, name=f'{label} - Heatmap', colormap=colormap, contrast_limits=contrast_limits, scale=(1, scale_factor, scale_factor))
    viewer.add_vectors(vector_grid, name=f'{label} - Vectors', vector_style=arrow_style,
                        edge_width=arrow_width, length=arrow_length, edge_color=arrow_edge_color, scale=(1, scale_factor, scale_factor))


# Add actual image
one_wave = tif.imread('/mnt/crunch/Clark/Fly_TFM/data/second/second_best_one_wave.tif')
viewer.add_image(one_wave, name='Actual Image', colormap='gray', contrast_limits=[0, 255])

# Colorbar display
viewer.layers['Raw Displacements - Heatmap'].colorbar.visible = True

# Side-by-side grid view (2 layers per box)
viewer.grid.enabled = True
viewer.grid.stride = 2

napari.run()